# QLoRA Fine-Tuning Gemma-4-2B on StackOverflow Python Q&A

This notebook fine-tunes the Gemma-4-2B model using QLoRA on StackOverflow Python questions and answers.

## 1. Setup and Installation

In [ ]:
!pip install transformers peft bitsandbytes accelerate datasets scipy unsloth torch -q

## 2. Load Processed Data from S3

In [ ]:
import boto3
import pandas as pd
from io import BytesIO

# S3 credentials: In production, use IAM roles or environment variables.
# AWS credentials are automatically loaded from:
# - Environment variables (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY)
# - ~/.aws/credentials file
# - IAM role (when running on EC2/ECS/Lambda)

# TODO: Set BUCKET_NAME to your project's S3 bucket (e.g., 'YOUR_NETID-so-python')
# Example: BUCKET_NAME = 'my_netid-so-python'
BUCKET_NAME = 'q1abc-so-python'  # Replace with your actual bucket name
s3_client = boto3.client('s3')

## 3. Load and Prepare Dataset

In [ ]:
from datasets import Dataset

# Convert to HuggingFace Datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

print(train_dataset)

In [ ]:
# Display a sample prompt
sample = train_dataset[0]
print("Sample prompt:")
print(sample['prompt'])

## 4. Load Gemma Model with QLoRA Configuration

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = 'google/gemma-4-2b'
MAX_SEQ_LENGTH = 512

# BitsAndBytes 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.model_max_length = MAX_SEQ_LENGTH

# Load model with QLoRA config
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto'
)

## 5. Training Configuration

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./gemma-qlora-finetuned',
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    warmup_steps=50,
    fp16=True,
    eval_strategy='steps',
    save_strategy='steps',
    load_best_model_at_end=True
)

## 6. Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    """Tokenize prompts and prepare labels."""
    result = tokenizer(
        examples['prompt'],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding='longest'
    )
    # Labels are the same as input_ids for causal LM
    result['labels'] = result['input_ids'].copy()
    return result

## 7. Train Model

In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM, not masked LM
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

# Train
trainer.train()

# Save model
trainer.save_model('./gemma-qlora-finetuned-final')

## 8. Export to GGUF for Ollama

In [ ]:
from peft import PeftModel
from transformers import AutoTokenizer

# Reload base model and merge LoRA weights
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto'
)

model = PeftModel.from_pretrained(base_model, './gemma-qlora-finetuned-final')
model = model.merge_and_unload()

# Save as HuggingFace format
model.save_pretrained('./gemma-qlora-gguf')
tokenizer.save_pretrained('./gemma-qlora-gguf')

print("Model exported to ./gemma-qlora-gguf")
print("To convert to GGUF for Ollama, use: llama-cli or llama.cpp converter")

## 9. Inference Comparison

In [ ]:
# Reload base model for comparison
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto'
)

fine_tuned_model = trainer.model